# Watermark localization: round v2 on Colab

Colab version of the JURECA round v2: synth-v2, then E171 (SegFormer-B2), kind gating and box refinement, then the submission.

**Before you start**
1. In Google Drive, create the folder `MyDrive/argos_wl/` and put `wl_code.zip` in it.
2. *Runtime → Change runtime type → GPU*. Pick an A100 if you can; an L4 also works. A T4 is slow, and the cells below shrink the run for it.
3. Optional, for uploading from here: under *Secrets* (key icon on the left), add `WMLOC_BASE_URL` and `WMLOC_API_KEY`, and turn on notebook access.
4. Run the cells from top to bottom. Each step skips work it already finished. After a disconnect, run everything again from the top.
5. At the end, the last cell writes `MyDrive/argos_wl/results.txt`. Send that file back.

Every step's output (`output/`, checkpoints, logs, the synth-v2 tar) is saved to Drive. Only the venv and the datasets are rebuilt after a disconnect.

In [ ]:
# 1. Drive, GPU check, budget knobs
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/argos_wl'
os.makedirs(f'{DRIVE}/output/logs', exist_ok=True)
assert os.path.isfile(f'{DRIVE}/wl_code.zip'), f'Upload wl_code.zip to {DRIVE} first'

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
assert gpu, 'No GPU: Runtime > Change runtime type > GPU'
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9

# Defaults suit an A100. JURECA plan: 6000 synth images per method, 30 epochs.
SYNTH_PER_METHOD, EPOCHS = 3000, 30
if 'L4' in gpu:
    EPOCHS = 20
elif 'T4' in gpu or 'V100' in gpu:
    SYNTH_PER_METHOD, EPOCHS = 1500, 12
WORKERS = max(1, min(6, os.cpu_count() - 1))
SYNTH_CACHE_GB = max(2, int(ram_gb * 0.4))   # RAM for decompressed synth shards

os.environ.update(DRIVE=DRIVE, REPO='/content/wl', GPU=gpu, N=str(SYNTH_PER_METHOD), EPOCHS=str(EPOCHS),
                  WORKERS=str(WORKERS), CACHE_GB=str(SYNTH_CACHE_GB))
print(f'GPU: {gpu} | CPUs: {os.cpu_count()} | RAM: {ram_gb:.0f} GB')
print(f'synth/method={SYNTH_PER_METHOD} epochs={EPOCHS} workers={WORKERS} synth_cache_gb={SYNTH_CACHE_GB}')

In [ ]:
%%bash
# 2. Code + data (about 5 min the first time: 570 MB from Hugging Face, 780 MB of COCO val2017)
set -euo pipefail
rm -rf /content/wl_src && unzip -q -o "$DRIVE/wl_code.zip" -d /content/wl_src
mkdir -p "$REPO" && cp -a /content/wl_src/watermarking-localization/. "$REPO/"   # new code, keeps data/.venv
cd "$REPO"
ln -sfn "$DRIVE/output" output

HF=https://huggingface.co/datasets/SprintML/Hackathon_Watermark_Localisation_Task/resolve/main
mkdir -p data/release data/clean
for f in train.zip test.zip starter_kit.zip; do [[ -f data/$f ]] || curl -sSfL -o data/$f $HF/$f; done
[[ -f data/release/train/metadata.json ]] || unzip -q data/train.zip -d data/release
[[ -f data/release/test/metadata.json ]] || unzip -q data/test.zip -d data/release
[[ -d kit/wmloc ]] || unzip -q data/starter_kit.zip -d kit
if [[ ! -d data/clean/val2017 ]]; then
  curl -sSfL -o data/val2017.zip http://images.cocodataset.org/zips/val2017.zip
  unzip -q data/val2017.zip -d data/clean && rm data/val2017.zip
fi
echo "train: $(ls data/release/train/images | wc -l) | test: $(ls data/release/test/images | wc -l) | coco: $(ls data/clean/val2017 | wc -l)"

In [ ]:
%%bash
# 3. Python 3.11 venv with the kit's pinned torch 2.5.1, then the kit models (about 5 min)
set -euo pipefail
cd "$REPO"
if [[ ! -x .venv/bin/python ]]; then
  pip -q install uv
  uv venv -q -p 3.11 .venv
  source .venv/bin/activate
  uv pip install -q -r kit/requirements.txt scipy matplotlib segmentation-models-pytorch pyyaml pilgram
fi
source .venv/bin/activate
python kit/download_models.py
python - <<'PY'
import sys, torch
sys.path.insert(0, 'kit')
from wmloc.watermarks import load_watermark
for m in ('trustmark', 'pixelseal', 'stegastamp'):
    load_watermark(m, models_dir='kit/models', device='cuda')
print('kit models OK | torch', torch.__version__, '| cuda', torch.cuda.is_available())
PY

In [ ]:
%%bash
# 4. synth-v2 (two workers on the GPU; roughly 30-60 min on an A100). Saved to Drive as synth_v2.tar.
set -euo pipefail
cd "$REPO" && source .venv/bin/activate
if [[ ! -f data/synth/v2.done && -f "$DRIVE/synth_v2.tar" ]]; then
  echo "restoring synth-v2 from Drive"; rm -rf data/synth/v2; mkdir -p data/synth; tar -xf "$DRIVE/synth_v2.tar" -C data
  touch data/synth/v2.done
fi
if [[ ! -f data/synth/v2.done ]]; then
  rm -rf data/synth/v2
  HALF=$(( (N + 1) / 2 )); PIDS=()
  for w in 0 1; do
    python -m wl.synth --out data/synth/v2 --num-per-method $HALF --shard-size 200 --train-folds 1 2 3 4 \
      --seed $w --shard-prefix shard_w$w --device cuda --clean-dir data/clean/val2017 \
      --external-prob 0.85 --fill-probs 0.6 0.3 0.1 > output/logs/synth_w$w.log 2>&1 &
    PIDS+=($!)
  done
  for p in "${PIDS[@]}"; do wait $p || { tail -20 output/logs/synth_w*.log; exit 1; }; done
  tar -cf "$DRIVE/synth_v2.tar" -C data synth/v2
  touch data/synth/v2.done
fi
echo "shards: $(ls data/synth/v2 | wc -l) | size: $(du -sh data/synth/v2 | cut -f1)"

**Step 5 trains E171.** It prints one line per epoch. Check how long epoch 1 takes: the whole run takes about `EPOCHS` times that.
If that would be more than about 4 h, stop the cell, lower `EPOCHS` in cell 1, rerun cell 1, then rerun this cell.

In [ ]:
%%bash
# 5. Train E171 (SegFormer-B2 on real + synth-v2). Writes output/E171_segb2_v2/ on Drive.
set -euo pipefail
cd "$REPO" && source .venv/bin/activate
if [[ -f output/E171_segb2_v2/test_probs.npz ]]; then echo "E171 already done"; exit 0; fi
python -m wl.train --config configs/E171_segb2_v2.yaml \
  train.epochs=$EPOCHS train.workers=$WORKERS data.synth_cache_gb=$CACHE_GB 2>&1 | tee output/logs/E171_colab.log

In [ ]:
%%bash
# 6. Kind gating + decoder box refinement (about 10-20 min), then per-method diagnosis
set -euo pipefail
cd "$REPO" && source .venv/bin/activate
python -m wl.refine --run output/E171_segb2_v2 --id E181_segb2_ref --box --device cuda --batch 32 2>&1 | tee output/logs/E181_refine.log
python -m wl.diagnose output/E171_segb2_v2 output/E181_segb2_ref 2>&1 | tee output/logs/diagnose.log

### Optional, only with time left (A100: about 2-3 h more)
Step 7 trains a second model (E172, W2-R34), then ensembles it with E171 and refines the ensemble. Skip it on a T4.

In [ ]:
%%bash
# 7. (optional) E172 + ensemble + refine
set -euo pipefail
cd "$REPO" && source .venv/bin/activate
if [[ ! -f output/E172_w2r34_v2/test_probs.npz ]]; then
  python -m wl.train --config configs/E172_w2r34_v2.yaml \
    train.workers=$WORKERS data.synth_cache_gb=$CACHE_GB 2>&1 | tee output/logs/E172_colab.log
fi
./scripts/runs/round4_ensemble.sh E190 output/E171_segb2_v2 output/E172_w2r34_v2 2>&1 | tee output/logs/E190_ens.log
python -m wl.refine --run output/E190_ens_logit --id E191_ens_ref --box --device cuda --batch 32 2>&1 | tee output/logs/E191_refine.log
python -m wl.diagnose output/E190_ens_logit output/E191_ens_ref 2>&1 | tee output/logs/diagnose_ens.log

In [ ]:
# 8. Pick the best run by OOF score and build its submission.pt (validated locally)
import json, os, subprocess
REPO = os.environ['REPO']
runs = ['E171_segb2_v2', 'E181_segb2_ref', 'E172_w2r34_v2', 'E190_ens_linear', 'E190_ens_logit', 'E190_ens_stack', 'E191_ens_ref']
scores = {}
for r in runs:
    p = f'{REPO}/output/{r}/metrics.json'
    if os.path.isfile(p) and os.path.isfile(f'{REPO}/output/{r}/test_probs.npz'):
        m = json.load(open(p))
        scores[r] = m.get('oof_mean') or m.get('mean')
for r, s in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f'{r:18s} OOF {s:.4f}')
BEST = max(scores, key=scores.get)
os.environ['BEST'] = BEST
print('\nbest:', BEST, '| JURECA E104 for comparison: 0.6258')
r = subprocess.run(['bash', '-c', 'cd $REPO && ./scripts/runs/submit.sh output/$BEST'], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])

In [ ]:
# 9. Upload to the leaderboard (needs the two Colab secrets). If this fails, download the file instead.
import os, subprocess
from google.colab import userdata, files
try:
    os.environ['WMLOC_BASE_URL'] = userdata.get('WMLOC_BASE_URL')
    os.environ['WMLOC_API_KEY'] = userdata.get('WMLOC_API_KEY')
    r = subprocess.run(['bash', '-c', 'cd $REPO && ./scripts/runs/submit.sh output/$BEST --submit'],
                       capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
except Exception as e:
    print('No upload:', e)
files.download(f"{os.environ['REPO']}/output/{os.environ['BEST']}/submission.pt")

In [ ]:
%%bash
# 10. Collect everything into MyDrive/argos_wl/results.txt. Send that file back.
cd "$REPO" && source .venv/bin/activate
{
  echo "== settings: GPU=$GPU synth/method=$N epochs=$EPOCHS workers=$WORKERS best=${BEST:-?}"
  python - <<'PY'
import json, os
from wl.evaluate import format_metrics_summary
for r in ['E171_segb2_v2', 'E181_segb2_ref', 'E172_w2r34_v2', 'E190_ens_linear', 'E190_ens_logit', 'E190_ens_stack', 'E191_ens_ref']:
    p = f'output/{r}/metrics.json'
    if os.path.isfile(p):
        m = json.load(open(p))
        print(f"{r}: {format_metrics_summary(m)} | best_epoch={m.get('best_epoch')} | OOF={m.get('oof_mean')}")
PY
  for f in E171_colab E181_refine diagnose E172_colab E190_ens E191_refine diagnose_ens synth_w0; do
    [[ -f output/logs/$f.log ]] || continue
    echo; echo "== $f"; grep -vE "it/s|s/it" output/logs/$f.log | tail -45
  done
} > "$DRIVE/results.txt" 2>&1
cat "$DRIVE/results.txt"